# Stance Training on Google Colab
Use a free Colab T4 to train the small two-stage stance pipeline with Drive-backed checkpoints and downloadable backups.


## Flow
- mount Drive
- clone repo
- install Colab-safe dependencies
- generate Colab-specific configs
- train `stage1_public_small` with step checkpoints
- zip/download a stage-1 backup any time
- train `stage2_hardcases_small`
- zip/download final model


In [ ]:
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/fact_checking_system_colab"

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
import os
import shutil

REPO_URL = "https://github.com/injetiharsha/fact_checking_system.git"
BRANCH = "feat/reduce-heuristics-phased"
REPO_DIR = "/content/fact_checking_system"

os.chdir("/content")

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


In [ ]:
!pip install -q --upgrade pip
!pip uninstall -y peft bitsandbytes sentence-transformers > /dev/null 2>&1 || true
!pip install -q   transformers==4.38.2   datasets==2.17.1   accelerate==0.27.2   scikit-learn==1.4.2   sentencepiece==0.2.0   PyYAML==6.0.2   tqdm==4.66.2


In [ ]:
!pip install -q --upgrade pip
!pip uninstall -y peft bitsandbytes

!pip install -q \
  transformers==4.38.2 \
  datasets==2.17.1 \
  accelerate==0.27.2 \
  scikit-learn==1.4.2 \
  sentencepiece==0.2.0 \
  PyYAML==6.0.2 \
  tqdm==4.66.2


In [ ]:
import os, torch
os.environ['PYTHONWARNINGS'] = 'ignore'
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Build the reduced public stage-1 dataset
!python training/common/build_stance_stage1_public.py


In [ ]:
# Create Colab-specific configs with larger batch, fp16, and frequent step checkpoints.
import os
from pathlib import Path
import yaml

base_stage1 = Path('training/configs/stance_stage1_public_small.yaml')
base_stage2 = Path('training/configs/stance_stage2_hardcases_small.yaml')
colab_stage1 = Path('training/configs/stance_stage1_public_small_colab.yaml')
colab_stage2 = Path('training/configs/stance_stage2_hardcases_small_colab.yaml')

with base_stage1.open('r', encoding='utf-8') as f:
    s1 = yaml.safe_load(f)
with base_stage2.open('r', encoding='utf-8') as f:
    s2 = yaml.safe_load(f)

if USE_DRIVE:
    os.makedirs(DRIVE_DIR, exist_ok=True)
    s1['data']['tokenized_cache_dir'] = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage1_public_small/tokenized_dataset')
    s1['output']['checkpoint_dir'] = os.path.join(DRIVE_DIR, 'checkpoints/stance/stage1_public_small')
    s1['output']['metrics_dir'] = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage1_public_small')
else:
    s1['data']['tokenized_cache_dir'] = 'training_artifacts/stance/stage1_public_small/tokenized_dataset'

s1['training']['batch_size'] = 32
s1['training']['eval_batch_size'] = 32
s1['training']['fp16'] = True
s1['training']['max_length'] = 256
s1['training']['logging_steps'] = 100
s1['training']['save_strategy'] = 'steps'
s1['training']['save_steps'] = 2000
s1['training']['evaluation_strategy'] = 'steps'
s1['training']['eval_steps'] = 2000
s1['training']['save_total_limit'] = 3
s1['training']['disable_tqdm'] = True

if USE_DRIVE:
    s2['data']['tokenized_cache_dir'] = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage2_hardcases_small/tokenized_dataset')
    s2['output']['checkpoint_dir'] = os.path.join(DRIVE_DIR, 'checkpoints/stance/stage2_hardcases_small')
    s2['output']['metrics_dir'] = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage2_hardcases_small')
    s2['model']['name'] = os.path.join(DRIVE_DIR, 'checkpoints/stance/stage1_public_small')

s2['training']['batch_size'] = 32
s2['training']['eval_batch_size'] = 32
s2['training']['fp16'] = True
s2['training']['max_length'] = 256
s2['training']['logging_steps'] = 20
s2['training']['save_strategy'] = 'steps'
s2['training']['save_steps'] = 50
s2['training']['evaluation_strategy'] = 'steps'
s2['training']['eval_steps'] = 50
s2['training']['save_total_limit'] = 3
s2['training']['disable_tqdm'] = True

with colab_stage1.open('w', encoding='utf-8') as f:
    yaml.safe_dump(s1, f, sort_keys=False)
with colab_stage2.open('w', encoding='utf-8') as f:
    yaml.safe_dump(s2, f, sort_keys=False)

print(colab_stage1.read_text())
print('---')
print(colab_stage2.read_text())


In [ ]:
# Optional helper: zip the latest stage-1 checkpoint at any point.
import os, shutil
from pathlib import Path

def zip_tree(src_dir, archive_base):
    src = Path(src_dir)
    if not src.exists():
        print('missing:', src)
        return None
    archive = shutil.make_archive(archive_base, 'zip', root_dir=src.parent, base_dir=src.name)
    print('created:', archive)
    return archive

def latest_checkpoint(base_dir):
    base = Path(base_dir)
    if not base.exists():
        return None
    checkpoints = sorted(base.glob('checkpoint-*'), key=lambda p: p.stat().st_mtime)
    return str(checkpoints[-1]) if checkpoints else str(base)


In [ ]:
# Stage 1 small on Colab T4
!python -u training/stance/train.py --config training/configs/stance_stage1_public_small_colab.yaml


In [ ]:
# Backup/download stage 1 immediately after it finishes.
import os, shutil
from google.colab import files

stage1_dir = os.path.join(DRIVE_DIR, 'checkpoints/stance/stage1_public_small') if USE_DRIVE else 'checkpoints/stance/stage1_public_small'
stage1_metrics = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage1_public_small') if USE_DRIVE else 'training_artifacts/stance/stage1_public_small'
archive = '/content/stance_stage1_public_small.zip'
!zip -r {archive} {stage1_dir} {stage1_metrics}
print('Created:', archive)
# files.download(archive)


In [ ]:
# Stage 2 hardcases small
!python -u training/stance/train.py --config training/configs/stance_stage2_hardcases_small_colab.yaml


In [ ]:
# Final backup/download
import os
from google.colab import files

final_dir = os.path.join(DRIVE_DIR, 'checkpoints/stance/stage2_hardcases_small') if USE_DRIVE else 'checkpoints/stance/stage2_hardcases_small'
metrics_dir = os.path.join(DRIVE_DIR, 'training_artifacts/stance/stage2_hardcases_small') if USE_DRIVE else 'training_artifacts/stance/stage2_hardcases_small'
archive = '/content/stance_stage2_hardcases_small.zip'
!zip -r {archive} {final_dir} {metrics_dir}
print('Created:', archive)
# files.download(archive)


## If your Colab session may end early
- because checkpoints are saved every few thousand steps to Drive, you keep progress even if the session ends
- you can run the zip cell after stage 1 to download an intermediate model
- you can also zip the latest `checkpoint-*` folder manually from Drive if needed
